In [1]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint, CSVLogger
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from data_utils import DataSet

import numpy as np
import os.path
import time

In [2]:
def load_cnn_model():
    base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(480, 640, 3), pooling='avg')
    
    print(base_model.summary())
    outputs = base_model.get_layer('global_average_pooling2d').output # 'avg_pool'

    cnn_model = Model(inputs=base_model.input, outputs=outputs)

    return cnn_model


In [3]:
cnn_model = load_cnn_model()

Model: "inception_v3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 480, 640, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 239, 319, 32)      │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 239, 319, 32)      │              96 │ conv2d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation (Activation)       │ (None, 239, 319, 32)      │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 237, 317, 32)      │           9,216 │ activation[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 237, 317, 32)      │              96 │ conv2d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_1 (Activation)     │ (None, 237, 317, 32)      │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 237, 317, 64)      │          18,432 │ activation_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 237, 317, 64)      │             192 │ conv2d_2[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_2 (Activation)     │ (None, 237, 317, 64)      │               0 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 118, 158, 64)      │               0 │ activation_2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 118, 158, 80)      │           5,120 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 118, 158, 80)      │             240 │ conv2d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_3 (Activation)     │ (None, 118, 158, 80)      │               0 │ batch_normalization_3[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_4 (Conv2D)             │ (None, 116, 156, 192)     │         138,240 │ activation_3[0][0]         │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 21,802,784 (83.17 MB)

 Trainable params: 21,768,352 (83.04 MB)

 Non-trainable params: 34,432 (134.50 KB)

None


# Creating Sequences

## Looks to be ~ 12 seconds per video

### ~24 minutes for 120 videos, ~6.6 hrs for entire 1980 videos

In [4]:
# ---- RUN THE FOLLOWING CELLS TO GENERATE .NPY SEQ FOR ALL VIDEOS IN DATASET ----- # 
#      SEQUENCES THAT ALREADY EXIST WILL BE SKIPPED

seq_length = 16
dataset = DataSet(cnn_model)

=== Inspeccionando los datos ===
Elemento 0: ['train', 'bandeja', 'Feans_Fran_RGB00001482'] | Longitud: 3
Elemento 1: ['train', 'bandeja', 'Feans_1_RGB00001064'] | Longitud: 3
Elemento 2: ['train', 'bandeja', 'Fas2_00001137_flipped_rotated'] | Longitud: 3
Elemento 3: ['train', 'bandeja', 'Feans_Pedro_RGB00000827_flipped'] | Longitud: 3
Elemento 4: ['train', 'bandeja', 'Anton00001901_rotated'] | Longitud: 3


In [5]:
dataset.data

[['train', 'bandeja', 'Feans_Fran_RGB00001482'],
 ['train', 'bandeja', 'Feans_1_RGB00001064'],
 ['train', 'bandeja', 'Fas2_00001137_flipped_rotated'],
 ['train', 'bandeja', 'Feans_Pedro_RGB00000827_flipped'],
 ['train', 'bandeja', 'Anton00001901_rotated'],
 ['train', 'bandeja', 'Lucas00001355_rotated'],
 ['train', 'bandeja', 'Merchi_RGB00000811_flipped_rotated'],
 ['train', 'bandeja', 'Martin00001019_flipped_rotated'],
 ['train', 'bandeja', 'Fas6_00002477_flipped_rotated'],
 ['train', 'bandeja', 'Feans_2_RGB00001477'],
 ['train', 'bandeja', 'Lucas00001466'],
 ['train', 'bandeja', 'Fas7_00001082_rotated'],
 ['train', 'bandeja', 'Anton00001980_rotated'],
 ['train', 'bandeja', 'Fas6_00002477'],
 ['train', 'bandeja', 'Anton00001980_flipped'],
 ['train', 'bandeja', 'Edu_RGB00001588'],
 ['train', 'bandeja', 'Feans_Pedro_RGB00000957_flipped_rotated'],
 ['train', 'bandeja', 'Anton00001980_flipped_rotated'],
 ['train', 'bandeja', 'Feans_Pedro_RGB00000827_flipped_rotated'],
 ['train', 'bandeja',

In [ ]:
tic = time.time()

for ind, sample in enumerate(dataset.data):
    
    path = os.path.join('data', 'sequences', sample[1], sample[2] + '-' + str(seq_length) + \
        '-features.npy')
    
    if os.path.isfile(path):
        print("Sequence: {} already exists".format(ind))
    else:
        print("Generating and saving sequence: {}".format(ind))
        sequence = dataset.extract_seq_features(sample)

print("Time Elapsed: {}".format(time.time() - tic))

# NOTE: The folders in `VIDEO_RGB` must be consolidated into the following folders in order for the data
# to retrieved properly according to the data_file.csv
# - backhand, bvolley, forehand, fvolley, service, smash
# OTHERWISE YOU WILL GET EMPTY SEQUENCES THAT CANNOT BE TRAINED ON

Sequence: 0 already exists
Sequence: 1 already exists
Generating and saving sequence: 2
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
Sequence: 3 already exists
Generating and saving sequence: 4
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/s

# Sanity Check on Generated Sequences

In [7]:
path = os.path.join('data', 'sequences', 'backhand', 'p36_backhand_s3-16-features.npy')

sequence = np.load(path)
sequence

FileNotFoundError: [Errno 2] No such file or directory: 'data\\sequences\\backhand\\p36_backhand_s3-16-features.npy'